# 🌞 Laboratorio 1: Modelos Autorregresivos Profundos (NN-AR)
## Caso de Estudio: Pronóstico del Ciclo Solar — Monthly Sunspots (1749–1983)

**Pontificia Universidad Javeriana Cali — Curso de Deep Learning**  
**Tutor:** Jan Polanco Velasco  
**Semestre:** 2026-2

---

### 🎯 Objetivo del Laboratorio

En este laboratorio construiremos una **Red Neuronal Autorregresiva (NN-AR)** para pronosticar la actividad de manchas solares mensuales. El dataset elegido — *Monthly Sunspots* — es un clásico de la física y el Machine Learning por las siguientes razones:

| Característica | Detalle |
|---|---|
| **Problema físico no lineal** | Las manchas solares siguen un ciclo cuasi-periódico de ~11 años con subidas abruptas y caídas lentas. Los modelos lineales (AR, ARIMA) fallan en capturar esta asimetría. |
| **Tamaño ideal** | 2,820 observaciones mensuales (1749–1983), permite FFT, descomposición estacional y entrenamiento en segundos en CPU. |
| **Cero dependencias locales** | Se descarga automáticamente desde GitHub, sin rutas frágiles. |

### 📋 Contenido

1. Carga y exploración de la serie temporal
2. Análisis espectral con Transformada Rápida de Fourier (FFT)
3. Partición temporal rigurosa (sin data leakage)
4. Construcción de la ventana deslizante (formulación autorregresiva)
5. Arquitectura de la Red Neuronal Multicapa (MLP)
6. Entrenamiento con detención temprana
7. Predicción a un paso (*one-step-ahead*)
8. Inferencia recursiva multi-paso (*recursive multi-step*)
9. Comparativa visual de resultados

---
## 1. 📚 Marco Teórico: Modelos Autorregresivos

### 1.1 ¿Qué es un modelo autorregresivo?

Un **modelo autorregresivo (AR)** asume que el valor actual de una serie temporal $T(k)$ puede explicarse como una función de sus valores pasados. Formalmente:

$$T(k) = f_{\theta}\big(T(k-1),\; T(k-2),\; \dots,\; T(k-n)\big) + \epsilon_k$$

donde:
- $n$ es el **orden del modelo** (cantidad de retardos o *lags*)
- $f_{\theta}$ es una función parametrizada por $\theta$ (en nuestro caso, una red neuronal)
- $\epsilon_k$ es el término de error (ruido blanco)

### 1.2 De serie temporal a problema supervisado

La idea central es transformar una secuencia temporal en **pares de entrenamiento** $(X, Y)$ usando una **ventana deslizante** (*sliding window*):

| Paso | Entrada $X$ | Salida $Y$ |
|------|-------------|------------|
| 1 | $[T_1, T_2, \dots, T_n]$ | $T_{n+1}$ |
| 2 | $[T_2, T_3, \dots, T_{n+1}]$ | $T_{n+2}$ |
| $\vdots$ | $\vdots$ | $\vdots$ |
| $m$ | $[T_m, T_{m+1}, \dots, T_{m+n-1}]$ | $T_{m+n}$ |

### 1.3 ¿Por qué una Red Neuronal y no un modelo AR clásico?

Un modelo AR clásico es **lineal**: $T(k) = \phi_1 T(k-1) + \phi_2 T(k-2) + \dots + \phi_n T(k-n) + c$

El problema es que el ciclo solar presenta:
- **Asimetría**: las subidas son más rápidas que las caídas
- **No linealidad**: la amplitud varía entre ciclos
- **Armoniciosas**: múltiples frecuencias superpuestas

Una **red neuronal multicapa (MLP)** puede aproximar cualquier función continua (Teorema de Aproximación Universal), capturando estas no linealidades.

### 1.4 Función de costo: Error Cuadrático Medio (MSE)

Para problemas de regresión, la función de costo más utilizada es el **MSE**:

$$\mathcal{L}_{MSE} = \frac{1}{N} \sum_{i=1}^{N} \big(y_i - \hat{y}_i\big)^2$$

Su gradiente respecto a los pesos $w$ permite actualizar la red mediante **gradiente descendente**:

$$w^{(t+1)} = w^{(t)} - \alpha \nabla_w \mathcal{L}(w^{(t)})$$

donde $\alpha$ es el *learning rate* (tasa de aprendizaje).

### 1.5 Normalización de datos

Antes de entrenar, normalizamos los datos usando **estandarización** (Z-score):

$$x_{\text{scaled}} = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$

> ⚠️ **Regla de oro**: El escalador se ajusta (*fit*) **únicamente** con los datos de entrenamiento para evitar *data leakage*.

---
## 2. ⚙️ Configuración Inicial

In [ ]:
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import tensorflow as tf
from plotly.subplots import make_subplots
from scipy.fft import fft, fftfreq
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.seasonal import seasonal_decompose
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Sequential

warnings.filterwarnings("ignore")

# Reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {len(tf.config.list_physical_devices('GPU')) > 0}")

---
## 3. 📥 Carga del Dataset

El dataset **Monthly Sunspots** contiene el número promedio de manchas solares observadas por mes, recopiladas desde 1749. Las manchas son regiones oscuras en la superficie del Sol causadas por actividad magnética intensa.

> 📖 **Referencia:** [1] Wolfer, A. (1892). *Zur Frage des Zusammenhangs zwischen Sonnentflecken und magnetischen Störungen der Erde*. — [2] [Brownlee, J. — Monthly Sunspots Dataset](https://raw.githubusercontent.com/jbrownlee/Datasets/master/monthly-sunspots.csv)

In [ ]:
URL_SUNSPOTS = (
    "https://raw.githubusercontent.com/jbrownlee/Datasets/master/monthly-sunspots.csv"
)

df = pd.read_csv(URL_SUNSPOTS)
df.columns = ["Fecha", "Sunspots"]
df["Fecha"] = pd.to_datetime(df["Fecha"])
df.set_index("Fecha", inplace=True)

print(f"Total observaciones: {len(df)} meses")
print(
    f"Período: {df.index.min().strftime('%Y-%m')} → {df.index.max().strftime('%Y-%m')}"
)
print(f"Duración: {(df.index.max() - df.index.min()).days / 365.25:.1f} años")
df.head(10)

### 3.1 Visualización de la serie temporal completa

Observamos los **ciclos de Schwabe** — el patrón cuasi-periódico de aproximadamente 11 años en la actividad solar. La línea naranja muestra la **media móvil de 12 meses**, que suaviza las fluctuaciones mensuales y revela la tendencia de cada ciclo.

In [ ]:
fig = go.Figure()

# Serie original
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["Sunspots"],
        mode="lines",
        line=dict(color="#94a3b8", width=0.8),
        name="Manchas Solares Mensuales",
        hovertemplate="%{x|%Y-%m}<br>Manchas: %{y:.0f}<extra></extra>",
    )
)

# Media móvil de 12 meses
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["Sunspots"].rolling(12).mean(),
        mode="lines",
        line=dict(color="#f59e0b", width=2.5),
        name="Media Móvil (12 meses)",
        hovertemplate="%{x|%Y-%m}<br>Promedio: %{y:.1f}<extra></extra>",
    )
)

fig.update_layout(
    title=dict(
        text="Actividad Solar Histórica (1749 – 1983): Ciclos de Schwabe (~11 años)",
        font=dict(size=16, color="#1e293b"),
    ),
    xaxis=dict(
        title="Año",
        gridcolor="#e2e8f0",
        showgrid=True,
        rangeslider=dict(visible=True, thickness=0.05),
    ),
    yaxis=dict(title="Número de Manchas Solares", gridcolor="#e2e8f0", zeroline=False),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        font=dict(size=11),
    ),
    height=500,
    margin=dict(l=60, r=30, t=80, b=40),
)

fig.show()

### 3.2 Estadísticas descriptivas

Analicemos las propiedades estadísticas básicas de la serie. Observa la **asimetría** (skewness): la distribución no es simétrica, lo cual confirma que un modelo lineal no sería ideal.

In [ ]:
stats = df["Sunspots"].describe()
stats["skewness"] = df["Sunspots"].skew()
stats["kurtosis"] = df["Sunspots"].kurtosis()
stats["variance"] = df["Sunspots"].var()
print("Estadísticas descriptivas de la serie temporal:")
print(stats.to_string())

---
## 4. 🔬 Análisis Espectral: Transformada Rápida de Fourier (FFT)

La **Transformada de Fourier** descompone una señal en sus componentes de frecuencia, permitiéndonos identificar los **periodos dominantes**. Matemáticamente:

$$X(f) = \sum_{k=0}^{N-1} x(k) \cdot e^{-j 2\pi f k / N}$$

El **periodo dominante** se obtiene como $T = 1/f_{\text{max}}$, donde $f_{\text{max}}$ es la frecuencia con mayor amplitud espectral.

> 🔎 **Interpretación física:** Un pico en $T \approx 132$ meses (11 años) confirma el **ciclo de Schwabe**.

In [ ]:
signal = df["Sunspots"].values
n_samples = len(signal)

# FFT: eliminar componente DC (promedio)
fft_vals = np.abs(fft(signal - np.mean(signal)))[: n_samples // 2]
frequencies = fftfreq(n_samples, d=1.0)[: n_samples // 2]  # d = 1 mes

# Convertir frecuencia → periodo (meses)
periods_in_months = 1.0 / frequencies[1:]  # omitir f=0 (DC)
amplitudes = fft_vals[1:]

# Periodo dominante
dominant_period = periods_in_months[np.argmax(amplitudes)]
print(
    f"Periodo dominante detectado por FFT: {dominant_period:.1f} meses ({dominant_period / 12:.1f} años)"
)
print("Esto corresponde al Ciclo de Schwabe (~11 años).")

In [ ]:
fig = go.Figure()

# Espectro completo
fig.add_trace(
    go.Scatter(
        x=periods_in_months,
        y=amplitudes,
        mode="lines",
        line=dict(color="#6366f1", width=1.8),
        fill="tozeroy",
        fillcolor="rgba(99, 102, 241, 0.1)",
        name="Magnitud espectral",
        hovertemplate="Periodo: %{x:.0f} meses<br>Amplitud: %{y:.0f}<extra></extra>",
    )
)

# Marcar el pico dominante
fig.add_vline(
    x=dominant_period,
    line_dash="dash",
    line_color="#ef4444",
    line_width=2,
    annotation_text=f"Pico dominante: {dominant_period:.0f} meses<br>({dominant_period / 12:.1f} años)",
    annotation_position="top right",
    annotation_font=dict(size=12, color="#ef4444"),
)

fig.update_layout(
    title=dict(
        text=f"Espectro de Frecuencias (FFT) — Ciclo de Schwabe detectado a ~{dominant_period / 12:.1f} años",
        font=dict(size=15, color="#1e293b"),
    ),
    xaxis=dict(title="Periodo (Meses)", range=[12, 300], gridcolor="#e2e8f0", dtick=24),
    yaxis=dict(title="Magnitud (Amplitud)", gridcolor="#e2e8f0", zeroline=False),
    template="plotly_white",
    height=450,
    margin=dict(l=60, r=30, t=80, b=40),
)

fig.show()

---
## 5. 📊 Descomposición Estacional

La **descomposición estacional** separa la serie en tres componentes:

$$T(k) = \underbrace{S(k)}_{\text{Estacional}} + \underbrace{T_r(k)}_{\text{Tendencia}} + \underbrace{R(k)}_{\text{Residuo}}$$

- **Tendencia ($T_r$):** Evolución de largo plazo de la actividad solar.
- **Estacional ($S$):** Patrón repetitivo del ciclo de ~11 años.
- **Residuo ($R$):** Lo que no explican los otros dos componentes (ruido + variabilidad irregular).

In [ ]:
# Periodo estacional = 132 meses (~11 años = Ciclo de Schwabe)
decomposition = seasonal_decompose(df["Sunspots"], model="additive", period=132)

fig = make_subplots(
    rows=4,
    cols=1,
    subplot_titles=[
        "Original",
        "Tendencia",
        "Estacional (Ciclo de ~11 años)",
        "Residuo",
    ],
    vertical_spacing=0.06,
    row_heights=[0.3, 0.25, 0.25, 0.2],
)

# Original
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=decomposition.observed,
        mode="lines",
        line=dict(color="#64748b", width=0.8),
        name="Original",
    ),
    row=1,
    col=1,
)

# Tendencia
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=decomposition.trend,
        mode="lines",
        line=dict(color="#f59e0b", width=2),
        name="Tendencia",
    ),
    row=2,
    col=1,
)

# Estacional
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=decomposition.seasonal,
        mode="lines",
        line=dict(color="#6366f1", width=1.2),
        name="Estacional",
    ),
    row=3,
    col=1,
)

# Residuo
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=decomposition.resid,
        mode="lines",
        line=dict(color="#ef4444", width=0.8),
        name="Residuo",
    ),
    row=4,
    col=1,
)

fig.update_layout(
    title=dict(
        text="Descomposición Estacional de la Serie de Manchas Solares",
        font=dict(size=15),
    ),
    template="plotly_white",
    height=700,
    showlegend=False,
    margin=dict(l=60, r=30, t=80, b=30),
)

for i in range(1, 5):
    fig.update_xaxes(gridcolor="#e2e8f0", row=i, col=1)
    fig.update_yaxes(gridcolor="#e2e8f0", row=i, col=1)

fig.show()

---
## 6. ✂️ Partición Temporal y Normalización

### ⚠️ Regla fundamental: ¡NUNCA muestreo aleatorio en series de tiempo

En series temporales, la **dependencia temporal** implica que los datos deben particionarse de forma **cronológica**. Un muestreo aleatorio causaría *data leakage* (fuga de información), ya que el modelo "vería" el futuro durante el entrenamiento.

```
|◄──── Entrenamiento (80%) ────►|◄──── Prueba (20%) ────►|
|  1749              ...   1934  |  1935           1983   |
```

### Normalización sin Data Leakage

$$x_{\text{scaled}} = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$

El escalador se ajusta **solo** con los datos de entrenamiento ($\mu_{\text{train}}$, $\sigma_{\text{train}}$) y luego se aplica a los datos de prueba con esos mismos parámetros.

In [ ]:
values = df["Sunspots"].values.reshape(-1, 1)

train_size = int(len(values) * 0.8)
train_raw = values[:train_size]
test_raw = values[train_size:]

print("Partición temporal (80% Train / 20% Test):")
print(f"  Entrenamiento: {len(train_raw)} meses ({train_size / 12:.1f} años)")
print(f"  Prueba:        {len(test_raw)} meses ({len(test_raw) / 12:.1f} años)")
print(f"  Corte temporal: {df.index[train_size].strftime('%Y-%m')}")

# Ajustar escalador SOLO con datos de entrenamiento
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_raw)
test_scaled = scaler.transform(test_raw)

print("\nParámetros del escalador (solo del train):")
print(f"  μ_train = {scaler.mean_[0]:.2f}")
print(f"  σ_train = {scaler.scale_[0]:.2f}")

---
## 7. 🧱 Construcción de la Ventana Deslizante

Transformamos la serie unidimensional en **pares supervisados** $(X, y)$:

$$\underbrace{\big[T(k-n), \; T(k-n+1), \; \dots, \; T(k-1)\big]}_{X \in \mathbb{R}^n} \;\xrightarrow{\;f_\theta\;}\; \underbrace{T(k)}_{y}$$

El hiperparámetro **window size** ($n$) define cuántos meses pasados usa el modelo para predecir el siguiente. En este laboratorio usamos $n = 24$ (2 años de memoria retrospectiva).

In [ ]:
WINDOW_SIZE = 24


def build_autoregressive_matrices(series, window_size):
    """Transforma serie unidimensional en pares (X: retardos, y: siguiente valor)."""
    X, y = [], []
    for i in range(len(series) - window_size):
        X.append(series[i : i + window_size, 0])
        y.append(series[i + window_size, 0])
    return np.array(X), np.array(y).reshape(-1, 1)


X_train, y_train = build_autoregressive_matrices(train_scaled, WINDOW_SIZE)
X_test, y_test = build_autoregressive_matrices(test_scaled, WINDOW_SIZE)

print(f"Formulación Autorregresiva con Window Size = {WINDOW_SIZE}:")
print(f"  X_train: {X_train.shape}  (muestras × retardos)")
print(f"  y_train: {y_train.shape}  (valor objetivo)")
print(f"  X_test:  {X_test.shape}")
print(f"  y_test:  {y_test.shape}")

In [ ]:
# Visualización del concepto de ventana deslizante
example_series = train_scaled[:30].flatten()

fig = go.Figure()

# Serie de ejemplo
fig.add_trace(
    go.Scatter(
        x=list(range(30)),
        y=example_series,
        mode="lines+markers",
        line=dict(color="#64748b", width=2),
        marker=dict(size=6, color="#64748b"),
        name="Serie normalizada",
    )
)

# Ventanas de ejemplo
colors = ["rgba(99,102,241,0.3)", "rgba(245,158,11,0.3)", "rgba(239,68,68,0.3)"]
border_colors = ["#6366f1", "#f59e0b", "#ef4444"]
labels = ["Ventana 1: X₁ → Y₁", "Ventana 2: X₂ → Y₂", "Ventana 3: X₃ → Y₃"]

for i, (start, c_fill, c_border, label) in enumerate(
    zip([0, 1, 2], colors, border_colors, labels)
):
    # Rectángulo de la ventana
    fig.add_vrect(
        x0=start,
        x1=start + WINDOW_SIZE - 0.5,
        fillcolor=c_fill,
        line_color=c_border,
        line_width=2,
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color=c_border),
    )
    # Flecha al objetivo
    fig.add_annotation(
        x=start + WINDOW_SIZE,
        y=example_series[start + WINDOW_SIZE],
        ax=start + WINDOW_SIZE - 1,
        ay=example_series[start + WINDOW_SIZE - 1] + 0.5,
        xref="x",
        yref="y",
        axref="x",
        ayref="y",
        showarrow=True,
        arrowhead=2,
        arrowsize=1.2,
        arrowcolor=c_border,
    )

fig.update_layout(
    title=f"Concepto de Ventana Deslizante (Window Size = {WINDOW_SIZE})",
    xaxis=dict(title="Mes (índice)", gridcolor="#e2e8f0"),
    yaxis=dict(title="Valor normalizado", gridcolor="#e2e8f0"),
    template="plotly_white",
    height=400,
    showlegend=False,
)

fig.show()

---
## 8. 🧠 Arquitectura de la Red Neuronal

### 8.1 Diseño de la MLP (Multilayer Perceptron)

La red sigue la arquitectura **piramidal** (reducción progresiva de neuronas):

$$\underbrace{n}_{\text{Entrada}} \;\rightarrow\; \underbrace{16}_{\text{Oculta 1 (ReLU)}} \;\rightarrow\; \underbrace{8}_{\text{Oculta 2 (ReLU)}} \;\rightarrow\; \underbrace{1}_{\text{Salida (Lineal)}}$$

### 8.2 Ecuación de cada neurona

Para una neurona $j$ en la capa $l$:

$$h_j^{[l]} = g\left( w_{j,0}^{[l]} + \sum_{i=1}^{n_{l-1}} w_{j,i}^{[l]} \cdot h_i^{[l-1]} \right)$$

donde $g(\cdot)$ es la función de activación:

| Activación | Fórmula | Uso |
|---|---|---|
| **ReLU** | $\max(0, x)$ | Capas ocultas (estándar) |
| **Lineal** | $x$ | Capa de salida en regresión |

### 8.3 Optimizador: Adam

**Adam** combina las ventajas de *Momentum* y *RMSprop*, ajustando la tasa de aprendizaje por parámetro:

$$w^{(t+1)} = w^{(t)} - \alpha \cdot \frac{\hat{m}^{(t)}}{\sqrt{\hat{v}^{(t)}} + \epsilon}$$

donde $\hat{m}$ y $\hat{v}$ son las estimaciones sesgadas del primer y segundo momento del gradiente.

In [ ]:
model = Sequential(
    [
        Input(shape=(WINDOW_SIZE,), name="Entradas_Retardos"),
        Dense(16, activation="relu", name="Capa_Oculta_1"),
        Dense(8, activation="relu", name="Capa_Oculta_2"),
        Dense(1, activation="linear", name="Salida_Lineal"),
    ],
    name="NN_Autorregresiva",
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.003), loss="mse", metrics=["mae"]
)

model.summary()

### 8.4 Visualización de la arquitectura

La siguiente tabla resume la arquitectura y el número de parámetros:

| Capa | Tipo | Neuronas | Activación | Parámetros |
|------|------|----------|------------|------------|
| Entrada | Dense | 24 | — | 0 |
| Oculta 1 | Dense | 16 | ReLU | $(24+1) \times 16 = 400$ |
| Oculta 2 | Dense | 8 | ReLU | $(16+1) \times 8 = 136$ |
| Salida | Dense | 1 | Lineal | $(8+1) \times 1 = 9$ |
| **Total** | | | | **545** |

---
## 9. 🏋️ Entrenamiento del Modelo

### 9.1 Detención temprana (*Early Stopping*)

Para evitar el **sobreajuste** (*overfitting*), monitoreamos la pérdida en validación. Si no mejora durante `patience` épocas, detenemos el entrenamiento y restauramos los mejores pesos:

$$\text{Si } \mathcal{L}_{\text{val}}(t) > \mathcal{L}_{\text{val}}(t-1) \;\forall\; t \in [t^*, t^* + \text{patience}], \;\text{detener y restaurar pesos de } t^*$$

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss", patience=15, restore_best_weights=True, verbose=1
)

# en train tengo 25 GB de imagenes, las paso 120 veces por la red neuronal (120 epochs), pasa 25 imagenes por batch (batch_size=25), y cada vez que pasa por todas las imagenes, se llama a una epoca (epoch)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=120,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1,
)

In [ ]:
# Curva de aprendizaje
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Función de Costo (MSE)", "Error Absoluto Medio (MAE)"],
)

# MSE
fig.add_trace(
    go.Scatter(
        y=history.history["loss"],
        mode="lines",
        line=dict(color="#4f46e5", width=2),
        name="Train MSE",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        y=history.history["val_loss"],
        mode="lines",
        line=dict(color="#ef4444", width=2),
        name="Val MSE",
    ),
    row=1,
    col=1,
)

# MAE
fig.add_trace(
    go.Scatter(
        y=history.history["mae"],
        mode="lines",
        line=dict(color="#4f46e5", width=2),
        name="Train MAE",
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        y=history.history["val_mae"],
        mode="lines",
        line=dict(color="#ef4444", width=2),
        name="Val MAE",
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig.update_xaxes(title_text="Época", gridcolor="#e2e8f0")
fig.update_yaxes(title_text="MSE (normalizado)", gridcolor="#e2e8f0", row=1, col=1)
fig.update_yaxes(title_text="MAE (normalizado)", gridcolor="#e2e8f0", row=1, col=2)

fig.update_layout(
    title=dict(text="Evolución del Entrenamiento", font=dict(size=15)),
    template="plotly_white",
    height=400,
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
)

fig.show()

---
## 10. 🎯 Predicción a Un Paso (*One-Step-Ahead Forecasting*)

En la predicción a un paso, el modelo recibe los **$n$ valores reales** más recientes y emite **una predicción** para el siguiente mes. Es la evaluación más directa del modelo:

$$\hat{T}(k) = f_\theta\big(T(k-1), T(k-2), \dots, T(k-n)\big)$$

### Métricas de evaluación

| Métrica | Fórmula | Interpretación |
|---------|---------|----------------|
| **RMSE** | $\sqrt{\frac{1}{N}\sum(y_i - \hat{y}_i)^2}$ | Error en las mismas unidades que la variable |
| **MAE** | $\frac{1}{N}\sum|y_i - \hat{y}_i|$ | Error promedio absoluto (robusta a outliers) |

In [ ]:
y_pred_1s_scaled = model.predict(X_test, verbose=0)

# Revertir la escala a unidades reales
y_test_real = scaler.inverse_transform(y_test)
y_pred_1s_real = scaler.inverse_transform(y_pred_1s_scaled)

mse_1step = np.mean((y_test_real - y_pred_1s_real) ** 2)
mae_1step = np.mean(np.abs(y_test_real - y_pred_1s_real))

print("Métricas de Pronóstico a 1 Paso:")
print(f"  RMSE: {np.sqrt(mse_1step):.2f} manchas solares")
print(f"  MAE:  {mae_1step:.2f} manchas solares")

---
## 11. 🔁 Inferencia Recursiva Multi-Paso

En la predicción recursiva, el modelo se **retroalimenta con sus propias predicciones** en lugar de usar valores reales. Esto simula un escenario real donde no tenemos acceso al futuro.

### Algoritmo recursivo

1. Tomar la ventana inicial $[T_{k-n}, \dots, T_{k-1}]$
2. Predecir $\hat{T}_k = f_\theta([T_{k-n}, \dots, T_{k-1}])$
3. Desplazar la ventana: eliminar $T_{k-n}$, agregar $\hat{T}_k$
4. Repetir desde el paso 2 para el siguiente horizonte

> ⚠️ **Problema:** El error se **acumula** con cada paso recursivo, degradando la calidad del pronóstico.

$$\text{Error acumulado} \propto \sqrt{H} \quad \text{(donde } H \text{ es el horizonte de predicción)}$$

In [ ]:
def recursive_forecast(trained_model, X_start, horizon):
    """Predicción recursiva: el modelo se retroalimenta con sus propias predicciones."""
    total_samples = len(X_start) - horizon + 1
    preds = []

    for i in range(0, total_samples, horizon):
        window = X_start[i].copy()
        block = []
        for _ in range(horizon):
            tensor_in = tf.convert_to_tensor(window.reshape(1, -1), dtype=tf.float32)
            next_val = float(trained_model(tensor_in, training=False).numpy()[0, 0])
            block.append(next_val)

            # Desplazar ventana: sale el más antiguo, entra la predicción
            window = np.roll(window, -1)
            window[-1] = next_val

        preds.extend(block)

    return np.array(preds).reshape(-1, 1)


# Predicción recursiva a 6 y 12 meses
y_pred_6s_scaled = recursive_forecast(model, X_test, horizon=6)
y_pred_12s_scaled = recursive_forecast(model, X_test, horizon=12)

# Revertir escala
y_pred_6s_real = scaler.inverse_transform(y_pred_6s_scaled)
y_pred_12s_real = scaler.inverse_transform(y_pred_12s_scaled)

# Métricas
lim_6 = len(y_pred_6s_real)
lim_12 = len(y_pred_12s_real)

rmse_6s = np.sqrt(np.mean((y_test_real[:lim_6] - y_pred_6s_real) ** 2))
rmse_12s = np.sqrt(np.mean((y_test_real[:lim_12] - y_pred_12s_real) ** 2))

print("RMSE Predicción Recursiva:")
print(f"  6 meses:  {rmse_6s:.2f} manchas")
print(f"  12 meses: {rmse_12s:.2f} manchas (error acumulado)")

---
## 12. 📊 Comparativa Visual de Resultados

Comparamos las tres estrategias de predicción contra los valores reales:

| Estrategia | Descripción | Ventaja | Desventaja |
|---|---|---|---|
| **1 Paso** | Usa valores reales en cada predicción | Máxima precisión | Necesita datos reales en tiempo real |
| **Recursiva 6m** | Se retroalimenta cada 6 meses | Pronóstico a mediano plazo | Error acumulado |
| **Recursiva 12m** | Se retroalimenta cada 12 meses | Pronóstico anual | Mayor error acumulado |

In [ ]:
# Zoom: 120 meses (1 ciclo solar completo en el conjunto de prueba)
ZOOM = 120

fig = go.Figure()

# Observaciones reales
fig.add_trace(
    go.Scatter(
        y=y_test_real[:ZOOM].flatten(),
        mode="lines+markers",
        name="Observaciones Reales",
        line=dict(color="#0f172a", width=2.2),
        marker=dict(size=3, color="#0f172a"),
        hovertemplate="Mes %{x}<br>Real: %{y:.1f}<extra></extra>",
    )
)

# 1 paso
fig.add_trace(
    go.Scatter(
        y=y_pred_1s_real[:ZOOM].flatten(),
        mode="lines",
        name=f"1 Paso (RMSE={np.sqrt(mse_1step):.1f})",
        line=dict(color="#2563eb", width=1.8, dash="dash"),
        hovertemplate="Mes %{x}<br>1 Paso: %{y:.1f}<extra></extra>",
    )
)

# Recursiva 6 meses
fig.add_trace(
    go.Scatter(
        y=y_pred_6s_real[:ZOOM].flatten(),
        mode="lines",
        name=f"Recursiva 6m (RMSE={rmse_6s:.1f})",
        line=dict(color="#f59e0b", width=1.8, dash="dot"),
        hovertemplate="Mes %{x}<br>Rec 6m: %{y:.1f}<extra></extra>",
    )
)

# Recursiva 12 meses
fig.add_trace(
    go.Scatter(
        y=y_pred_12s_real[:ZOOM].flatten(),
        mode="lines",
        name=f"Recursiva 12m (RMSE={rmse_12s:.1f})",
        line=dict(color="#ef4444", width=2.2),
        hovertemplate="Mes %{x}<br>Rec 12m: %{y:.1f}<extra></extra>",
    )
)

fig.update_layout(
    title=dict(
        text=f"Comparativa de Pronóstico Autorregresivo — 1 Ciclo Solar Completo ({ZOOM} meses de prueba)",
        font=dict(size=15, color="#1e293b"),
    ),
    xaxis=dict(title="Meses consecutivos de prueba", gridcolor="#e2e8f0"),
    yaxis=dict(title="Número de Manchas Solares", gridcolor="#e2e8f0", zeroline=False),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.03,
        xanchor="center",
        x=0.5,
        font=dict(size=11),
    ),
    height=520,
    margin=dict(l=60, r=30, t=90, b=40),
)

fig.show()

In [ ]:
# Gráfico de error: distribución de residuos
residuals_1step = y_test_real.flatten() - y_pred_1s_real.flatten()

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Distribución del Error (1 Paso)", "Error vs Valor Real"],
)

# Histograma de residuos
fig.add_trace(
    go.Histogram(
        x=residuals_1step,
        nbinsx=40,
        marker_color="#6366f1",
        opacity=0.75,
        name="Residuos",
    ),
    row=1,
    col=1,
)

# Línea vertical en cero
fig.add_vline(x=0, line_dash="dash", line_color="#ef4444", line_width=2, row=1, col=1)

# Error vs valor real
fig.add_trace(
    go.Scatter(
        x=y_test_real.flatten(),
        y=residuals_1step,
        mode="markers",
        marker=dict(size=4, color="#6366f1", opacity=0.5),
        name="Residuos vs Real",
    ),
    row=1,
    col=2,
)

fig.add_hline(y=0, line_dash="dash", line_color="#ef4444", line_width=2, row=1, col=2)

fig.update_xaxes(gridcolor="#e2e8f0")
fig.update_yaxes(gridcolor="#e2e8f0")

fig.update_layout(
    title=dict(text="Análisis de Residuos — Predicción a 1 Paso", font=dict(size=14)),
    template="plotly_white",
    height=400,
    showlegend=False,
)

fig.update_xaxes(title_text="Error (manchas)", row=1, col=1)
fig.update_yaxes(title_text="Frecuencia", row=1, col=1)
fig.update_xaxes(title_text="Valor Real (manchas)", row=1, col=2)
fig.update_yaxes(title_text="Error", row=1, col=2)

fig.show()

---
## 13. 📈 Resumen de Métricas

| Estrategia | RMSE (manchas) | Observación |
|---|---|---|
| Predicción 1 Paso | $\sqrt{\text{MSE}_{1s}}$ | Referencia óptima (usa valores reales) |
| Recursiva 6 Meses | $\text{RMSE}_{6s}$ | Error moderado por acumulación |
| Recursiva 12 Meses | $\text{RMSE}_{12s}$ | Mayor error (todo recursivo) |

> 🔎 **Conclusión pedagógica:** A mayor horizonte de predicción recursiva, mayor es la degradación del pronóstico. Esto demuestra la importancia de la **calidad de los datos de entrada** y el problema del **error acumulado** en modelos autorregresivos.

In [ ]:
# Tabla resumen como gráfico de barras
estrategias = ["1 Paso", "Recursiva 6m", "Recursiva 12m"]
rmse_vals = [np.sqrt(mse_1step), rmse_6s, rmse_12s]
colores = ["#2563eb", "#f59e0b", "#ef4444"]

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=estrategias,
        y=rmse_vals,
        marker_color=colores,
        text=[f"{v:.1f}" for v in rmse_vals],
        textposition="outside",
        textfont=dict(size=14, color="#1e293b"),
        hovertemplate="%{x}<br>RMSE: %{y:.2f} manchas<extra></extra>",
    )
)

fig.update_layout(
    title=dict(
        text="Comparación de RMSE por Estrategia de Predicción", font=dict(size=15)
    ),
    xaxis=dict(title="Estrategia"),
    yaxis=dict(title="RMSE (manchas solares)", gridcolor="#e2e8f0"),
    template="plotly_white",
    height=420,
    showlegend=False,
)

fig.show()

---
## 14. 🎓 Conclusiones

1. **Los modelos autorregresivos neuronales (NN-AR)** capturan exitosamente la no linealidad del ciclo solar, algo que un modelo AR clásico no lograría.

2. **La FFT** confirma matemáticamente el periodo dominante de ~11 años (Ciclo de Schwabe), validando la estructura del dataset.

3. **La predicción a 1 paso** ofrece la mejor precisión porque utiliza valores reales del pasado, pero en la práctica necesitamos predicciones multi-paso.

4. **La inferencia recursiva** degrada su precisión con el horizonte, demostrando el problema del **error acumulado** en series de tiempo.

5. **La ventana deslizante** de 24 meses (2 años) proporciona suficiente contexto histórico para que la red aprenda la dinámica del ciclo.

### 🚀 Próximos pasos

- Explorar **ventanas más grandes** (36, 48 meses) para capturar más contexto.
- Implementar arquitecturas **recurrentes** (LSTM, GRU) que manejan memoria de forma explícita.
- Comparar con **ARIMA** y otros modelos estadísticos clásicos.
- Agregar **features exógenas** (actividad magnética del Sol, etc.).

---

**📚 Referencias:**

[1] Wolfer, A. (1892). *Zur Frage des Zusammenhangs zwischen Sonnentflecken und magnetischen Störungen der Erde*.  
[2] [Brownlee, J. — Monthly Sunspots Dataset](https://raw.githubusercontent.com/jbrownlee/Datasets/master/monthly-sunspots.csv)  
[3] Hyndman, R.J. & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice*. 3rd ed. OTexts.  
[4] Chollet, F. (2021). *Deep Learning with Python*. 2nd ed. Manning Publications.